# Eval on one class — watch the other classes' cluster weights move (real ImageNet)

A **configurable** test run of cluster-reparameterized metagradient data curation
(`README.md`, hypothesis **H4**), on a **subset of real ImageNet classes**. The held-out *objective*
`phi` is the masked-reconstruction loss on a **single target class** `c*`. We run signed-gradient
**metagradient descent** on the per-class cluster-weight logits `theta` (exact metagradient via
`replay.py`'s REPLAY engine) and **record the whole weight vector `w = softmax(theta / tau)` at every
meta-step** — so we can see how the weights of *the other classes* redistribute as the method curates
the training pool toward `c*`.

**Expected story (H4).** `c*`'s own cluster and visually-related classes get **upweighted**;
orthogonal classes get **downweighted**. The default subset is the README's running example —
**target = timber wolf**, with other canids (fox, coyote, husky, shepherd, retriever) as near
neighbours and vehicles (sports car, school bus, fire engine) as orthogonal distractors.

Pick any subset of the 1000 ImageNet classes in the `ExperimentConfig` block below (by index or by a
unique name substring). Data is read lazily from the memory-mapped Arrow shards of
`benjamin-paine/imagenet-1k-128x128`; the full dataset and full label columns are never materialized.

> **Runtime & what to expect.** Defaults are tuned for a **24 GB NVIDIA L4 + 100 GB host RAM**:
> all **1000 ImageNet classes**, GPU-memory REPLAY checkpoints, strict IEEE FP32, **10,000-step**
> differentiable REPLAY training runs, and three **10,000-step** deep-training comparisons. The
> metagradient config is 64x64; the
> metagradient loop is extremely compute-intensive because backward-over-backward REPLAY recomputes
> each long trajectory at every meta-step. A high-memory CUDA GPU and fast checkpoint storage are
> strongly recommended; lower the scale knobs (§7) for smoke tests.


## 1. Configuration — *edit this one cell*

In [ ]:
from __future__ import annotations

import math
import os
from dataclasses import dataclass, field

import matplotlib.pyplot as plt
import numpy as np
import torch

from mae import MaeConfig, MaskedAutoencoderViT, make_patch_mask
from functional_train import SmoothAdamWConfig, initialize_train_state
from metagrad import InnerBatch, ObjectiveBatch
from replay import ReplayCheckpointConfig, replay_metagradient


@dataclass(frozen=True)
class ExperimentConfig:
    # --- reproducibility / hardware -------------------------------------------------
    seed: int = 0
    device: str = "auto"         # "auto" (CUDA if available) | "cpu" | "cuda"
    replay_tf32: bool = False     # keep disabled: strict IEEE FP32 matmuls are required for stability

    # --- dataset + class subset (the clusters / D' basis) ---------------------------
    dataset_name: str = "benjamin-paine/imagenet-1k-128x128"
    offline: bool = False         # read only the local HF cache (no network)
    train_split: str = "train"          # training-pool images come from here
    objective_split: str = "validation"  # held-out c* objective images (disjoint -> no leakage)
    # Keep only labels [0, num_labels) from each split.
    num_labels: int = 1000
    target: object = 0           # c* must be one of the retained labels
    include_target_in_pool: bool = True
    images_per_class: int = 100   # metagrad pool: 1000 x 100 @ 64px ~ 4.6 GiB (lives in host RAM)
    num_objective_images: int = 50
    keep_metagrad_pool_on_device: bool = False  # host pool -> ~13 GiB VRAM for REPLAY; per-batch PCIe copies are negligible vs compute

    # --- model / masking ------------------------------------------------------------
    image_size: int = 64         # ImageNet resized to this; 64px/patch8 -> 64 patches
    patch_size: int = 8
    encoder_dim: int = 256
    encoder_depth: int = 12
    encoder_heads: int = 8
    decoder_dim: int = 256
    decoder_depth: int = 8
    decoder_heads: int = 4
    mlp_ratio: float = 2.0
    mask_ratio: float = 0.5

    # --- inner training trajectory (the differentiable A) ---------------------------
    inner_steps: int = 3_000     # T: ~7.7 epochs over the 100k pool; solid inner convergence within the L4 budget
    batch_size: int = 256        # L4-sized differentiable batch; lower if backward-over-backward OOMs
    mask_seed: int = 7
    eval_mask_seed: int = 99     # fixed deterministic objective mask

    # --- inner optimizer (differentiable smooth AdamW) ------------------------------
    learning_rate: float = 5e-4
    beta1: float = 0.9
    beta2: float = 0.99
    eps: float = 1e-4
    weight_decay: float = 0.0

    # --- meta-optimization: theta <- theta - alpha_k * sign(grad) -------------------
    temperature: float = 0.5     # tau
    meta_steps: int = 20         # weights plateau by ~step 18 with alpha_decay=0.90 (within the L4 budget)
    alpha0: float = 0.25         # initial signed step
    alpha_decay: float = 0.90    # faster geometric anneal so sign-descent settles inside meta_steps
    branching_factor: int = 24   # measured ~6.2 GiB peak checkpoint states at T=3000; low recompute
    replay_checkpoint_backend: str = "memory"  # L4 24 GB VRAM can retain the lazy tree without disk I/O
    replay_checkpoint_directory: str | None = None
    replay_checkpoint_interval: int | None = None

    # --- deep training runs (strong standard recipe; NOT the differentiable loop) ----
    deep_steps: int = 10_000     # optimizer steps for EACH weighted/unweighted deep run
    deep_batch: int = 256         # conservative FP32 standard-training batch for the L4
    deep_lr: float = 1e-3        # peak LR (linear warmup -> cosine decay to eta_min)
    warmup_steps: int = 500
    eta_min: float = 1e-5
    deep_weight_decay: float = 0.05
    deep_betas: tuple = (0.9, 0.95)
    grad_clip: float = 1.0
    eval_every: int = 250        # held-out phi cadence
    image_every: int = 1000      # reconstruction-panel / sample-histogram cadence
    num_workers: int = 8         # tune to the VM's vCPU count and image-decoding throughput
    sampler_candidate_batch: int = 8192  # bounded label reads for lazy rejection sampling

    # --- Weights & Biases logging ---------------------------------------------------
    wandb_project: str = "metagrad-cluster-curation"
    wandb_entity: object = None  # str | None (your team/user; None = default)
    wandb_mode: str = "online"   # "online" | "offline" | "disabled"
    wandb_group: object = None   # str | None (None -> timestamped group)


cfg = ExperimentConfig()
replay_checkpoint_config = ReplayCheckpointConfig(
    backend=cfg.replay_checkpoint_backend,
    directory=cfg.replay_checkpoint_directory,
    interval_steps=cfg.replay_checkpoint_interval,
)
cfg

In [ ]:
# Enforce strict IEEE FP32 everywhere. BF16/FP16 autocast and TF32 are deliberately disabled.
torch_dtype = torch.float32
torch.set_default_dtype(torch_dtype)
assert not cfg.replay_tf32, "strict FP32 requires replay_tf32=False"

if cfg.device == "auto":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
else:
    device = torch.device(cfg.device)

if device.type == "cuda":
    from determinism import assert_replay_determinism, configure_replay_determinism
    configure_replay_determinism(cfg.seed, tf32=cfg.replay_tf32)
    assert_replay_determinism(tf32=False)
    cuda_props = torch.cuda.get_device_properties(device)
    print(f"GPU={cuda_props.name}  VRAM={cuda_props.total_memory / 2**30:.1f} GiB  tf32={cfg.replay_tf32}")
    if cuda_props.total_memory < 20 * 2**30:
        print("warning: this L4-tuned profile expects about 24 GiB VRAM; lower batch_size/branching_factor")
else:
    torch.manual_seed(cfg.seed)

assert torch.get_default_dtype() == torch.float32
print(f"device={device}  dtype={torch_dtype}  autocast=disabled")

In [ ]:
# --- Weights & Biases (online) -----------------------------------------------------------------
# Comprehensive logging for the metagrad loop and the deep training runs. Online mode needs
# `wandb login` on this machine; the import is guarded so a missing install/login degrades to a
# no-op instead of crashing. All runs share one `wandb_group` so W&B overlays them.
import time

try:
    import wandb
    _WANDB_AVAILABLE = True
except Exception as _exc:                       # pragma: no cover - environment dependent
    wandb, _WANDB_AVAILABLE = None, False
    print(f"wandb unavailable ({_exc!r}); logging disabled")

_WANDB_ON = _WANDB_AVAILABLE and cfg.wandb_mode != "disabled"
wandb_group = cfg.wandb_group or f"cwm-{time.strftime('%Y%m%d-%H%M%S')}"


class _NullRun:
    """No-op run so logging stays safe when wandb is unavailable or disabled."""

    def __init__(self):
        self.summary, self.name, self.id = {}, "disabled", None

    def log(self, *args, **kwargs):
        pass

    def finish(self, *args, **kwargs):
        pass


def start_run(name, config, job_type):
    if not _WANDB_ON:
        return _NullRun()
    return wandb.init(
        project=cfg.wandb_project, entity=cfg.wandb_entity, group=wandb_group,
        job_type=job_type, name=name, mode=cfg.wandb_mode, config=config, reinit=True,
    )


print(f"wandb: available={_WANDB_AVAILABLE}  mode={cfg.wandb_mode}  group={wandb_group}")

## 2. Load the class subset from real ImageNet

Images come from the local cache (offline, non-streaming). For each selected class we take the first
`images_per_class` training images; the **objective set** is fresh `c*` images from the *validation*
split, so train/eval are strictly disjoint (`README.md` §4.1). Classes are balanced, so the base
distribution `D` is uniform (`theta = 0` -> `w = 1/k`).

The `class_selection` / `target` entries may be class **indices** or unique **name substrings**
(resolved against the dataset's label names via `label_mapping.py`).


In [ ]:
mae_config = MaeConfig(
    image_size=cfg.image_size, patch_size=cfg.patch_size,
    encoder_dim=cfg.encoder_dim, encoder_depth=cfg.encoder_depth, encoder_heads=cfg.encoder_heads,
    decoder_dim=cfg.decoder_dim, decoder_depth=cfg.decoder_depth, decoder_heads=cfg.decoder_heads,
    mlp_ratio=cfg.mlp_ratio, mask_ratio=cfg.mask_ratio,
)

if cfg.offline:
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["HF_DATASETS_OFFLINE"] = "1"

from datasets import load_dataset
from torchvision.transforms import v2

ds = load_dataset(cfg.dataset_name, keep_in_memory=False)
class_names = ds[cfg.train_split].features["label"].names
short = lambda c: class_names[c].split(",")[0]


def resolve_spec(spec) -> int:
    """An int index, or a case-insensitive substring that uniquely matches one class name."""
    if isinstance(spec, (int, np.integer)):
        i = int(spec)
        if not 0 <= i < len(class_names):
            raise ValueError(f"class index {i} out of range 0..{len(class_names) - 1}")
        return i
    matches = [i for i, n in enumerate(class_names) if str(spec).lower() in n.lower()]
    if not matches:
        raise ValueError(f"no class name contains {spec!r}")
    if len(matches) > 1:
        opts = ", ".join(f"{i}:{short(i)}" for i in matches)
        raise ValueError(f"{spec!r} is ambiguous -> {opts}  (use an index or a longer substring)")
    return matches[0]


target_class = resolve_spec(cfg.target)
if not 0 <= target_class < cfg.num_labels:
    raise ValueError(f"target label {target_class} must be in [0, {cfg.num_labels})")

pool_classes = list(range(cfg.num_labels))
if not cfg.include_target_in_pool:
    pool_classes.remove(target_class)
num_groups = len(pool_classes)
assert num_groups >= 2, "need at least two retained labels"
pool_index = {c: j for j, c in enumerate(pool_classes)}

image_transform = v2.Compose([
    v2.ToImage(),
    v2.Resize((cfg.image_size, cfg.image_size), antialias=True),
    v2.ToDtype(torch.float32, scale=True),
])


def find_first_indices(split: str, counts_by_label: dict[int, int]) -> dict[int, list[int]]:
    """Scan the label-only Arrow view, retaining only the bounded set of requested row indices."""
    found = {label: [] for label in counts_by_label}
    remaining = sum(counts_by_label.values())
    label_rows = ds[split].select_columns(["label"])
    for row_index, row in enumerate(label_rows):
        label = int(row["label"])
        if label in found and len(found[label]) < counts_by_label[label]:
            found[label].append(row_index)
            remaining -= 1
            if remaining == 0:
                break
    if remaining:
        missing = {label: counts_by_label[label] - len(rows) for label, rows in found.items()
                   if len(rows) < counts_by_label[label]}
        raise ValueError(f"split {split!r} is missing requested examples: {missing}")
    return found


def decode_indices(split: str, rows: list[int]) -> torch.Tensor:
    """Decode only explicitly selected rows; never materialize an entire split."""
    return torch.stack([image_transform(ds[split][row]["image"].convert("RGB")) for row in rows])


train_rows = find_first_indices(
    cfg.train_split, {class_id: cfg.images_per_class for class_id in pool_classes}
)
images_list, group_list = [], []
for c in pool_classes:
    images_list.append(decode_indices(cfg.train_split, train_rows[c]))
    group_list += [pool_index[c]] * cfg.images_per_class
# The current L4 profile keeps the ~2.5 GB metagrad pool on-device to avoid repeating the same
# host-to-device copies during REPLAY recomputation. Disable the knob for smaller GPUs.
pool_device = device if cfg.keep_metagrad_pool_on_device else torch.device("cpu")
training_images = torch.cat(images_list).to(device=pool_device, dtype=torch_dtype)
group_ids = torch.tensor(group_list, dtype=torch.long, device=pool_device)
base_group_masses = torch.full((num_groups,), 1.0 / num_groups, dtype=torch_dtype, device=device)

objective_rows = find_first_indices(cfg.objective_split, {target_class: cfg.num_objective_images})
objective_cpu = decode_indices(cfg.objective_split, objective_rows[target_class])
objective_images = objective_cpu.to(device=device, dtype=torch_dtype)
objective_mask = make_patch_mask(
    torch.arange(900_000, 900_000 + cfg.num_objective_images),
    step=0, seed=cfg.eval_mask_seed,
    num_patches=mae_config.num_patches, mask_ratio=cfg.mask_ratio,
).to(device)
objective_batch = ObjectiveBatch(objective_images, objective_mask)
assert training_images.dtype == objective_batch.images.dtype == torch.float32


In [ ]:
# Pixel-space "visual similarity to c*" -- a dependency-free proxy for the low-level reconstruction
# transfer that drives the metagradient (used only to order/colour the §5-§6 plots, not by the run
# itself). Compare each class's mean image *after subtracting the cross-class mean*: raw natural-image
# means are all ~0.97 cosine-similar (a shared brown/green blur), so centering is what exposes the
# class-distinctive structure. Cheap even at k=1000 (1000 x 12288 floats, on CPU).
class_means = torch.stack([imgs.mean(0).flatten() for imgs in images_list])  # [num_groups, D], cpu
centered = class_means - class_means.mean(0, keepdim=True)
target_centered = (centered[pool_index[target_class]] if cfg.include_target_in_pool
                   else (objective_cpu.mean(0).flatten() - class_means.mean(0)))
visual_sim_to_target = torch.nn.functional.cosine_similarity(
    centered, target_centered.unsqueeze(0), dim=1
)

# Quick sanity peek: the nearest/farthest classes to c* in this proxy (the H4 ordering axis).
_order = torch.argsort(visual_sim_to_target, descending=True)
print(f"visual-similarity proxy to c* = {short(target_class)} (centered mean-image cosine):")
print("  nearest :", ", ".join(f"{short(pool_classes[j])} {visual_sim_to_target[j]:+.2f}"
                               for j in _order[:6].tolist()))
print("  farthest:", ", ".join(f"{short(pool_classes[j])} {visual_sim_to_target[j]:+.2f}"
                               for j in _order[-6:].tolist()))

### Peek at the data

A couple of images per class so you can see what the model (and the curation) is actually working
with at this resolution.


In [ ]:
# With k up to 1000 we can't draw every cluster -- show a small capped sample (target first).
show_classes = list(pool_classes[:8])
if target_class in pool_classes and target_class not in show_classes:
    show_classes = [target_class] + show_classes[:7]
n_show = min(3, cfg.images_per_class)
fig, axes = plt.subplots(len(show_classes), n_show,
                         figsize=(1.4 * n_show, 1.4 * len(show_classes)))
axes = np.atleast_2d(axes)
for r, c in enumerate(show_classes):
    j = pool_index[c]
    for s in range(n_show):
        ax = axes[r, s]
        ax.imshow(images_list[j][s].permute(1, 2, 0).numpy())
        ax.set_xticks([]); ax.set_yticks([])
    label = short(c) + ("  (c*)" if c == target_class else "")
    axes[r, 0].set_ylabel(label, fontsize=8, rotation=0, ha="right", va="center")
fig.suptitle(f"Training pool sample ({len(show_classes)} of {num_groups} classes)", y=0.99)
fig.tight_layout()
plt.show()

## 3. Model + deterministic inner trajectory

`theta = 0` -> uniform `w = 1/k` -> the base distribution `D`. Each of the `T` inner steps trains on
a deterministic minibatch with a fresh seeded MAE mask (bit-deterministic, no augmentation), exactly the
persistent cluster-softmax weighting the REPLAY engine differentiates through.


In [ ]:
model = MaskedAutoencoderViT(mae_config).to(device=device, dtype=torch_dtype)
initial_state = initialize_train_state(model)
assert all(value.dtype == torch.float32 for value in initial_state.parameters.values())
assert all(value.dtype == torch.float32 for value in initial_state.first_moments.values())
assert all(value.dtype == torch.float32 for value in initial_state.second_moments.values())

# Minibatch inner trajectory: one fixed, seeded shuffle of the whole pool, then one slice
# per inner step. The L4 profile keeps the pool on GPU; smaller-GPU profiles can keep it on CPU.
# Only the current minibatch enters the differentiable step, so the full T-step trajectory is
# never materialized. The shuffle is deterministic (seeded CPU generator) and `trajectory[t]`
# is identical on the forward pass and on every REPLAY recompute, so the determinism GATE
# still holds. The MAE mask stays keyed by GLOBAL image id + step -- a pure function of
# identity, exactly as in the full-batch version.
N = training_images.shape[0]
shuffle = torch.Generator().manual_seed(cfg.seed)
perm = torch.randperm(N, generator=shuffle)          # global indices, fixed for the whole run


def make_batch(t: int) -> InnerBatch:
    start = (t * cfg.batch_size) % N
    idx = perm[start:start + cfg.batch_size]
    if idx.numel() < cfg.batch_size:                 # wrap the tail -> keep batch size fixed
        idx = torch.cat([idx, perm[:cfg.batch_size - idx.numel()]])
    image_idx = idx.to(training_images.device)
    group_idx = idx.to(group_ids.device)
    mask = make_patch_mask(
        idx, step=t, seed=cfg.mask_seed,
        num_patches=mae_config.num_patches, mask_ratio=cfg.mask_ratio,
    )
    return InnerBatch(
        training_images.index_select(0, image_idx).to(device=device, dtype=torch_dtype),
        mask.to(device),
        group_ids.index_select(0, group_idx).to(device),
    )



class DeterministicInnerTrajectory:
    """Rebuild fixed minibatches on demand instead of retaining all T batches in RAM/VRAM."""

    def __len__(self):
        return cfg.inner_steps

    def __getitem__(self, step):
        if not isinstance(step, int) or not 0 <= step < len(self):
            raise IndexError(step)
        return make_batch(step)


trajectory = DeterministicInnerTrajectory()
optimizer_config = SmoothAdamWConfig(
    learning_rate=cfg.learning_rate, betas=(cfg.beta1, cfg.beta2),
    eps=cfg.eps, weight_decay=cfg.weight_decay,
)
tree_depth = math.ceil(math.log(cfg.inner_steps + 1, cfg.branching_factor))
views = cfg.inner_steps * cfg.batch_size
state_bytes = 3 * sum(p.numel() * p.element_size() for p in model.parameters())
checkpoint_bound = 1 + (cfg.branching_factor - 1) * tree_depth
print(f"model params = {sum(p.numel() for p in model.parameters()):,}  |  T = {cfg.inner_steps} inner steps")
print(f"batch_size = {cfg.batch_size}  ->  {views:,} example-views over the run "
      f"({views / N:.2f} epochs of the {N:,}-image pool)")
print(f"REPLAY branching factor = {cfg.branching_factor}  ->  tree depth {tree_depth}")
print(f"conservative checkpoint-state bound = {checkpoint_bound} states / "
      f"{checkpoint_bound * state_bytes / 2**30:.1f} GiB")

## 4. Metagradient descent — record every class weight

`theta <- theta - alpha_k * sign(grad_theta phi)`, with the exact metagradient recomputed by REPLAY
each meta-step and a geometric anneal on the step size. We log `phi` and the full weight vector
`w = softmax(theta / tau)` at every step.


In [ ]:
theta = torch.zeros(num_groups, device=device, dtype=torch_dtype, requires_grad=True)
assert theta.dtype == torch.float32

uniform_w = 1.0 / num_groups
target_j = pool_index.get(target_class)   # column of c* in w (None if held out of the pool)

meta_run = start_run(
    name="metagrad",
    job_type="metagrad",
    config=dict(phase="metagrad", num_groups=num_groups, target=int(target_class),
                target_name=short(target_class), meta_steps=cfg.meta_steps, temperature=cfg.temperature,
                alpha0=cfg.alpha0, alpha_decay=cfg.alpha_decay, inner_steps=cfg.inner_steps,
                batch_size=cfg.batch_size, image_size=cfg.image_size, patch_size=cfg.patch_size,
                branching_factor=cfg.branching_factor, learning_rate=cfg.learning_rate,
                replay_checkpoint_backend=cfg.replay_checkpoint_backend,
                replay_checkpoint_interval=cfg.replay_checkpoint_interval,
                replay_tf32=cfg.replay_tf32, dtype="float32"),
)

weight_history, phi_history, grad_history = [], [], []
for k in range(cfg.meta_steps + 1):
    # replay_metagradient shows its own per-step "REPLAY metagradient" progress bar (replay.py).
    phi, grad = replay_metagradient(
        model, initial_state, trajectory, objective_batch,
        theta, base_group_masses, optimizer_config, cfg.temperature,
        branching_factor=cfg.branching_factor,
        checkpoint_config=replay_checkpoint_config,
    )
    w = torch.softmax(theta.detach() / cfg.temperature, dim=0).cpu()
    g = grad.detach().cpu()
    weight_history.append(w)
    phi_history.append(phi.item())
    grad_history.append(g)

    alpha_k = cfg.alpha0 * cfg.alpha_decay ** k
    entropy = float(-(w * (w + 1e-12).log()).sum())
    order = torch.argsort(w, descending=True)
    metrics = {
        "meta/phi": phi.item(),
        "meta/metagrad_norm": float(g.norm()),
        "meta/metagrad_l1": float(g.abs().sum()),
        "meta/alpha_k": alpha_k,
        "meta/grad_sign_pos_frac": float((g > 0).float().mean()),
        "meta/w_entropy": entropy,
        "meta/w_entropy_frac": entropy / math.log(num_groups),
        "meta/w_kl_to_uniform": float((w * ((w + 1e-12).log() - math.log(uniform_w))).sum()),
        "meta/w_max": float(w.max()),
        "meta/w_above_uniform": int((w > uniform_w).sum()),
    }
    if target_j is not None:
        metrics["meta/target_weight"] = float(w[target_j])
        metrics["meta/target_rank"] = int((order == target_j).nonzero().item()) + 1
    if _WANDB_ON:
        metrics["meta/w_hist"] = wandb.Histogram(w.numpy())
        if k == cfg.meta_steps:                    # one labeled bar of the final top weights
            topk = order[: min(15, num_groups)].tolist()
            tbl = wandb.Table(data=[[short(pool_classes[j]), float(w[j])] for j in topk],
                              columns=["class", "weight"])
            metrics["meta/top_weights"] = wandb.plot.bar(tbl, "class", "weight",
                                                         title="Final top cluster weights")
    meta_run.log(metrics, step=k)
    print(f"meta-step {k:>2}/{cfg.meta_steps}   phi = {phi.item():.6f}", flush=True)

    if k < cfg.meta_steps:
        with torch.no_grad():
            theta -= alpha_k * grad.sign()

weight_history = torch.stack(weight_history)   # [meta_steps + 1, num_groups]
grad_history = torch.stack(grad_history)
meta_axis = list(range(cfg.meta_steps + 1))

meta_run.summary.update(dict(phi_initial=phi_history[0], phi_final=phi_history[-1],
                             phi_delta=phi_history[-1] - phi_history[0]))
meta_run.finish()
print(f"phi (held-out {short(target_class)} recon MSE): "
      f"{phi_history[0]:.6f} -> {phi_history[-1]:.6f}  (delta {phi_history[-1] - phi_history[0]:+.6f})")

## 5. How the (other) classes' weights move

Each line is one cluster's weight over meta-steps, coloured by **visual similarity to `c*`** (yellow =
similar, purple = orthogonal); the target is the black dashed line. Watch the visually-related classes
rise above the uniform `1/k` start and the orthogonal ones fall.


In [ ]:
sims = visual_sim_to_target
norm = (sims - sims.min()) / (sims.max() - sims.min() + 1e-9)

fig, ax = plt.subplots(figsize=(9, 5))
for j, c in enumerate(pool_classes):
    if c == target_class:
        ax.plot(meta_axis, weight_history[:, j], color="black", lw=2.6, ls="--",
                label=f"{short(c)} (c*)", zorder=5)
    else:
        ax.plot(meta_axis, weight_history[:, j], color=plt.cm.viridis(float(norm[j])),
                lw=1.8, label=short(c))
ax.axhline(1.0 / num_groups, color="grey", ls=":", lw=1, label="uniform 1/k")
ax.set_xlabel("meta-step")
ax.set_ylabel("cluster weight  w_i")
ax.set_title(f"Cluster weights over metagradient descent (objective = {short(target_class)} only)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=4, fontsize=8, frameon=False)
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

In [ ]:
print("% weight increase")
for a, b in zip(delta[order].numpy(), labels):
    print(a,b)

In [ ]:
w0, wT = weight_history[0], weight_history[-1]
delta = wT - w0
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# (A) objective over meta-steps
axes[0].plot(meta_axis, phi_history, "o-", color="C0")
axes[0].set(xlabel="meta-step", ylabel="phi (held-out MSE)",
            title=f"Objective: held-out {short(target_class)} reconstruction")
axes[0].grid(alpha=0.25)

# (B) net weight movement per class, sorted
order = torch.argsort(delta, descending=True)
labels = [short(pool_classes[j]) + (" *" if pool_classes[j] == target_class else "") for j in order]
bar_colors = ["#2ca02c" if delta[j] >= 0 else "#d62728" for j in order]
axes[1].bar(range(num_groups), delta[order].numpy(), color=bar_colors)
axes[1].axhline(0, color="grey", lw=0.8)
axes[1].set_xticks(range(num_groups))
axes[1].set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
axes[1].set(ylabel="w_final - w_initial", title="Net weight movement (green up / red down)")
axes[1].grid(alpha=0.25, axis="y")

# (C) final weight vs visual similarity to c* (H4 proxy)
axes[2].scatter(sims.numpy(), wT.numpy(), color="C2", zorder=3)
for j, c in enumerate(pool_classes):
    axes[2].annotate(short(c), (float(sims[j]), float(wT[j])), fontsize=7,
                     xytext=(3, 3), textcoords="offset points")
axes[2].axhline(1.0 / num_groups, color="grey", ls=":", label="uniform 1/k")
axes[2].set(xlabel="pixel-space visual similarity to c*  (proxy)", ylabel="final weight w_i",
            title="Final weight vs. visual similarity (H4)")
axes[2].legend(fontsize=8)
axes[2].grid(alpha=0.25)

fig.tight_layout()
plt.show()

## 6. Summary

In [ ]:
ranking = torch.argsort(wT, descending=True)
print("Final cluster weights (most upweighted first):")
for rank, j in enumerate(ranking.tolist()):
    c = pool_classes[j]
    tag = "   <-- TARGET c*" if c == target_class else ""
    print(f"  #{rank + 1:>2}  class {c:>4}  vis-sim {sims[j]:+.3f}  "
          f"w = {wT[j]:.4f}   delta = {wT[j] - w0[j]:+.4f}   {short(c)}{tag}")

print(f"\nphi: {phi_history[0]:.6f} -> {phi_history[-1]:.6f}  "
      f"(delta {phi_history[-1] - phi_history[0]:+.6f})")
if float(wT.std()) > 0 and float(sims.std()) > 0:
    corr = float(np.corrcoef(wT.numpy(), sims.numpy())[0, 1])
    print(f"corr(final weight, visual similarity to c*) = {corr:+.3f}   "
          "(near +1 => visually similar classes upweighted, consistent with H4)")
if cfg.include_target_in_pool:
    tj = pool_index[target_class]
    rank_of_target = int((ranking == tj).nonzero().item()) + 1
    print(f"target c* = {short(target_class)}: final weight {wT[tj]:.4f}  (rank {rank_of_target} of {num_groups})")

## 7. Knobs to try

- **Different subset / target:** edit `class_selection` and `target` (indices or unique name
  substrings, e.g. `target="timber wolf"`, or add `"tabby"`, `"airliner"`, ...). Browse names with
  `from label_mapping import idx2label`.
- **Hold the target out entirely:** set `include_target_in_pool=False` to watch *only the other
  classes* compete to best reconstruct an unseen `c*` — the purest "how do the others move" view.
- **Stronger signal:** raise `images_per_class`, `inner_steps` (`T`), or `image_size` (must stay
  divisible by `patch_size`); lower `temperature` for peakier `w`. Bigger configs are much faster on
  `device="cuda"`.
- **Class coverage:** this run intentionally keeps all 1000 ImageNet classes. REPLAY cost is set
  mainly by `T` and model size, not by the number of class-weight metaparameters.
- **Precision:** this notebook intentionally enforces strict IEEE FP32 everywhere. BF16/FP16
  autocast and TF32 are disabled because the metagradient and long ordinary-training runs become
  numerically unstable at reduced precision.


## 8. Deep training on the curated dataset (1000 classes, 64px) — sampling vs loss vs none

Now we *actually train* models with a **strong, standard recipe** (not the differentiable inner
loop): fresh `MaskedAutoencoderViT`, real `torch.optim.AdamW` (betas 0.9/0.95, weight-decay 0.05),
linear warmup → cosine decay, gradient clipping, batch `deep_batch`, `deep_steps` steps, vectorized
MAE masking. Data is read through a standard map-style `DataLoader` over **all 1000 ImageNet
classes**. Arrow shards remain memory-mapped on storage; workers decode only the current batch,
and samplers read labels in bounded chunks without building a full label or per-image-weight array.

The curated cluster weights `w = softmax(theta / tau)` from §4 enter exactly one of two ways, while
the third run is the no-curation baseline:

- **weighted · sampling** — a lazy rejection sampler draws images with probability ∝ `w_{class}`
  (the curated distribution `D'`) without full-dataset weights; plain reconstruction loss.
- **weighted · loss** — uniform sampling, but each example's loss is scaled by its cluster weight
  (`weighted_example_loss` with the learned `theta`) — the curation's original mechanism.
- **unweighted** — uniform sampling + plain loss (base `D`).

All three start from an identical init/seed; only the weighting differs. Everything is logged to
**Weights & Biases** (per-step loss / lr / grad-norm / throughput, periodic held-out φ, reconstruction
panels, sampled-class histograms) and overlaid via the shared run group.

> Heads-up: this is real training — ~3 × `deep_steps` × `deep_batch` image decodes, so wall-clock is
> dataloader-bound. Tune `num_workers` (use **0 on Windows**). Lower `deep_steps` for a smoke test.
> The 10,000-step metagrad trajectory covers the 1000-class pool repeatedly. On the L4, the lazy
> REPLAY tree stays in GPU memory with branching factor 24, avoiding disk I/O while using the extra
> VRAM to reduce recomputation. Lower `batch_size` or `branching_factor` if the VM reports OOM.

In [ ]:
from torch.utils.data import Dataset, DataLoader, Sampler
from mae import (patchify, unpatchify, masked_reconstruction_loss,
                 per_example_masked_reconstruction_loss)
from weighting import weighted_example_loss


# ---- standard map-style dataset over disk-backed Arrow shards (decoded per batch) ----------------
class ImagenetImages(Dataset):
    def __init__(self, hf_split, transform):
        self.split, self.transform = hf_split, transform

    def __len__(self):
        return len(self.split)

    def __getitem__(self, i):
        ex = self.split[int(i)]
        return self.transform(ex["image"].convert("RGB")), int(ex["label"])


deep_dataset = ImagenetImages(ds[cfg.train_split], image_transform)
deep_labels = ds[cfg.train_split].select_columns(["label"])

# label -> cluster column (identity when every class is in the pool); -1 marks a dropped class.
label_to_group = torch.full((len(class_names),), -1, dtype=torch.long)
for _j, _c in enumerate(pool_classes):
    label_to_group[_c] = _j


class LazyClassSampler(Sampler):
    """Sample map-style rows lazily using bounded label reads and rejection sampling."""

    def __init__(self, labels, class_weights, num_samples, seed, candidate_batch):
        weights = torch.as_tensor(class_weights, dtype=torch.float32, device="cpu")
        if weights.shape != (len(class_names),) or not torch.isfinite(weights).all():
            raise ValueError("class_weights must be one finite weight per dataset class")
        if (weights < 0).any() or not (weights > 0).any():
            raise ValueError("class_weights must be nonnegative with at least one positive entry")
        if int(num_samples) < 1 or int(candidate_batch) < 1:
            raise ValueError("num_samples and candidate_batch must be positive")
        self.labels = labels
        self.acceptance = weights / weights.max()
        self.num_samples = int(num_samples)
        self.seed = int(seed)
        self.candidate_batch = int(candidate_batch)

    def __len__(self):
        return self.num_samples

    def __iter__(self):
        generator = torch.Generator().manual_seed(self.seed)
        yielded = 0
        while yielded < self.num_samples:
            candidates = torch.randint(len(self.labels), (self.candidate_batch,), generator=generator)
            labels = torch.as_tensor(self.labels[candidates.tolist()]["label"], dtype=torch.long)
            keep = torch.rand(self.candidate_batch, generator=generator) < self.acceptance[labels]
            for index in candidates[keep].tolist():
                yield index
                yielded += 1
                if yielded == self.num_samples:
                    return

DEEP = dict(steps=cfg.deep_steps, batch=cfg.deep_batch, lr=cfg.deep_lr, warmup=cfg.warmup_steps,
            weight_decay=cfg.deep_weight_decay, betas=tuple(cfg.deep_betas), grad_clip=cfg.grad_clip,
            eval_every=cfg.eval_every, image_every=cfg.image_every, num_workers=cfg.num_workers,
            sampler_candidate_batch=cfg.sampler_candidate_batch,
            eta_min=cfg.eta_min)


def random_patch_mask(batch_size, num_patches, mask_ratio, generator=None):
    """Vectorized MAE mask (True = hidden); each row hides exactly floor(num_patches*ratio)."""
    num_masked = int(num_patches * mask_ratio)
    ids = torch.rand(batch_size, num_patches, generator=generator).argsort(dim=1)
    mask = torch.zeros(batch_size, num_patches, dtype=torch.bool)
    return mask.scatter_(1, ids[:, :num_masked], True)


def _cosine_lr(step):
    if step < DEEP["warmup"]:
        return DEEP["lr"] * (step + 1) / DEEP["warmup"]
    prog = (step - DEEP["warmup"]) / max(1, DEEP["steps"] - DEEP["warmup"])
    return DEEP["eta_min"] + 0.5 * (DEEP["lr"] - DEEP["eta_min"]) * (1.0 + math.cos(math.pi * prog))


@torch.no_grad()
def _eval_phi(net):
    net.eval()
    pred = net(objective_batch.images, objective_batch.patch_mask)
    val = masked_reconstruction_loss(objective_batch.images, pred, objective_batch.patch_mask).item()
    net.train()
    return val


@torch.no_grad()
def _recon_panel(net, n=6):
    net.eval()
    imgs, msk = objective_batch.images[:n], objective_batch.patch_mask[:n]
    pred = net(imgs, msk)
    net.train()
    targets = patchify(imgs, cfg.patch_size)
    completed = torch.where(msk.unsqueeze(-1), pred, targets)
    masked = targets.clone(); masked[msk] = 0.5
    orig = unpatchify(targets, cfg.patch_size, cfg.image_size).clamp(0, 1)
    hole = unpatchify(masked, cfg.patch_size, cfg.image_size).clamp(0, 1)
    recon = unpatchify(completed, cfg.patch_size, cfg.image_size).clamp(0, 1)
    rows = [torch.cat([orig[i], hole[i], recon[i]], dim=2) for i in range(n)]
    panel = torch.cat(rows, dim=1).permute(1, 2, 0).float().cpu().numpy()
    return wandb.Image(panel, caption="rows: orig | masked | reconstruction")


def train_strong(run_name, class_sampling_weights, loss_logits, seed, *, weighted_sampling):
    """Strong from-scratch MAE training through a finite, lazy, map-style DataLoader."""
    torch.manual_seed(seed)
    net = MaskedAutoencoderViT(mae_config).to(device=device, dtype=torch_dtype)
    assert all(parameter.dtype == torch.float32 for parameter in net.parameters())
    opt = torch.optim.AdamW(net.parameters(), lr=DEEP["lr"], betas=DEEP["betas"],
                            weight_decay=DEEP["weight_decay"])
    sampler = LazyClassSampler(
        deep_labels, class_sampling_weights, DEEP["steps"] * DEEP["batch"], seed,
        DEEP["sampler_candidate_batch"],
    )
    loader = DataLoader(deep_dataset, batch_size=DEEP["batch"], sampler=sampler,
                        num_workers=DEEP["num_workers"], pin_memory=(device.type == "cuda"),
                        drop_last=True, persistent_workers=DEEP["num_workers"] > 0)
    mask_gen = torch.Generator().manual_seed(seed + 1)
    run = start_run(run_name, {**DEEP, "run": run_name, "weighted_loss": loss_logits is not None,
                               "weighted_sampling": weighted_sampling,
                               "image_size": cfg.image_size, "patch_size": cfg.patch_size,
                               "mask_ratio": cfg.mask_ratio, "num_groups": num_groups,
                               "target_name": short(target_class)}, job_type="deep-train")

    phi_curve, phi_steps = [], []
    seen, t0 = 0, time.time()
    net.train()
    for step, (images, labels) in enumerate(loader):
        images = images.to(device=device, dtype=torch_dtype, non_blocking=True)
        assert images.dtype == torch.float32
        mask = random_patch_mask(images.shape[0], mae_config.num_patches, cfg.mask_ratio,
                                 generator=mask_gen).to(device)
        lr = _cosine_lr(step)
        for pg in opt.param_groups:
            pg["lr"] = lr
        opt.zero_grad(set_to_none=True)
        pred = net(images, mask)
        per_ex = per_example_masked_reconstruction_loss(images, pred, mask)
        if loss_logits is None:
            loss = per_ex.mean()
        else:
            gids = label_to_group[labels].to(device)
            loss = weighted_example_loss(per_ex, loss_logits, gids, base_group_masses, cfg.temperature)
        loss.backward()
        gnorm = torch.nn.utils.clip_grad_norm_(net.parameters(), DEEP["grad_clip"])
        opt.step()
        seen += images.shape[0]

        log = {"train/loss": float(loss), "train/lr": lr, "train/grad_norm": float(gnorm),
               "train/images_per_sec": seen / max(1e-9, time.time() - t0)}
        if step % DEEP["eval_every"] == 0 or step == DEEP["steps"] - 1:
            val = _eval_phi(net); phi_curve.append(val); phi_steps.append(step)
            log["eval/phi"] = val
        if _WANDB_ON and (step % DEEP["image_every"] == 0 or step == DEEP["steps"] - 1):
            log["eval/reconstructions"] = _recon_panel(net)
            log["sample/label_hist"] = wandb.Histogram(labels.float().numpy())
        run.log(log, step=step)
        if step % 500 == 0 or step == DEEP["steps"] - 1:
            print(f"[{run_name}] step {step:>5}/{DEEP['steps']}  loss {float(loss):.4f}  "
                  f"lr {lr:.2e}  phi {phi_curve[-1]:.5f}", flush=True)

    run.summary.update(dict(phi_initial=phi_curve[0], phi_final=phi_curve[-1],
                            phi_delta=phi_curve[-1] - phi_curve[0]))
    run.finish()
    return dict(phi_curve=phi_curve, phi_steps=phi_steps, net=net)

In [ ]:
# (A) weight through SAMPLING -- draw rows lazily with acceptance proportional to w[class].
# These vectors contain one entry per class, never one entry per image.
uniform_class_sampling_weights = torch.zeros(len(class_names), dtype=torch_dtype)
uniform_class_sampling_weights[pool_classes] = 1.0
weighted_class_sampling_weights = torch.zeros(len(class_names), dtype=torch_dtype)
weighted_class_sampling_weights[pool_classes] = weight_history[-1].to(torch_dtype)

result_sampling = train_strong("deep-weighted-sampling", weighted_class_sampling_weights, None,
                               seed=cfg.seed, weighted_sampling=True)

In [ ]:
# (B) weight through the LOSS -- uniform sampling, but scale each example's loss by its cluster
# weight via weighted_example_loss with the learned theta (the curation's original mechanism).
result_loss = train_strong("deep-weighted-loss", uniform_class_sampling_weights,
                           theta.detach().to(device=device, dtype=torch_dtype), seed=cfg.seed,
                           weighted_sampling=False)

In [ ]:
# (C) baseline -- uniform sampling, plain loss (base distribution D, no curation).
result_unweighted = train_strong("deep-unweighted", uniform_class_sampling_weights, None,
                                 seed=cfg.seed, weighted_sampling=False)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for res, lbl, col in [(result_unweighted, "unweighted (base D)", "C0"),
                      (result_sampling, "weighted · sampling (D')", "C1"),
                      (result_loss, "weighted · loss", "C2")]:
    ax.plot(res["phi_steps"], res["phi_curve"], "-", color=col, label=lbl)
ax.set_xlabel("training step")
ax.set_ylabel(f"phi: held-out {short(target_class)} recon MSE")
ax.set_title(f"Deep training ({DEEP['steps']} steps, {cfg.image_size}px): sampling vs loss vs none")
ax.legend(); ax.grid(alpha=0.25); fig.tight_layout(); plt.show()

b = result_unweighted["phi_curve"][-1]
s = result_sampling["phi_curve"][-1]
l = result_loss["phi_curve"][-1]
print(f"final phi   unweighted={b:.6f}   sampling={s:.6f}   loss={l:.6f}")
print(f"vs unweighted   sampling {s - b:+.6f}   loss {l - b:+.6f}   (negative = curation helps)")